In [ ]:
import requests
import pandas as pd
from datetime import datetime

# 5 tỉnh lớn ĐBSCL
provinces = {
    "An Giang": (10.5216, 105.1259),
    "Kien Giang": (10.0125, 105.0809),
    "Dong Thap": (10.4938, 105.6882),
    "Long An": (10.6956, 106.2431),
    "Can Tho": (10.0452, 105.7469)
}

start_date = "2021-01-01"
end_date = datetime.today().strftime("%Y-%m-%d")

all_data = []

for province, (lat, lon) in provinces.items():

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max",
        "timezone": "Asia/Bangkok"
    }

    response = requests.get(url, params=params)
    data = response.json()

    df = pd.DataFrame(data["daily"])

    df = df.rename(columns={
        "time": "Date",
        "temperature_2m_mean": "AvgTemp_C",
        "temperature_2m_max": "MaxTemp_C",
        "temperature_2m_min": "MinTemp_C",
        "precipitation_sum": "Rainfall_mm",
        "windspeed_10m_max": "WindSpeed_kmh"
    })

    df["Province"] = province
    df["Date"] = pd.to_datetime(df["Date"])

    all_data.append(df)

# Gộp dữ liệu
final_df = pd.concat(all_data)

# Reset index cho sạch
final_df = final_df.reset_index(drop=True)

# Sắp xếp
final_df = final_df.sort_values(["Province", "Date"])

# Xuất file JSON
final_df.to_json(
    "weather_dbscl_2021_2026.json",
    orient="records",
    force_ascii=False,
    indent=4,
    date_format="iso"
)

print("DONE! File JSON đã tạo.")
print("Số dòng:", final_df.shape[0])

DONE! File JSON đã tạo.
Số dòng: 9350


In [ ]:
from google.colab import drive
drive.mount('/content/drive')